In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

### GloFAS — Global Flood Awareness System (Copernicus EWDS)

**GloFAS** provides daily river discharge estimates worldwide based on the LISFLOOD model forced with ERA5 reanalysis.

- **Variable:** `dis24` — river discharge in the last 24 hours (m³/s)
- **Spatial resolution:** 0.05° (~5 km)
- **Temporal coverage:** 1979–present

**Prerequisites:** `cdsapi`, `~/.cdsapirc` with EWDS credentials.

> This notebook uses **synthetic demo data** that reproduces the GloFAS NetCDF format without requiring CDS credentials.
> Replace the synthetic section with actual `download_glofas()` calls when credentials are available.

In [2]:
# ipyleaflet is NOT installed — interactive map skipped
# from pyhydra.utils import interactive_map  # requires ipyleaflet

from pyhydra.data_sources.river_discharge import (
    download_glofas,
    download_glofas_by_year,
    read_glofas_nc,
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr
import tempfile, os
from pathlib import Path
print('Imports OK')

Imports OK


---
## Download configuration (real use)

When real CDS credentials are available, configure `~/.cdsapirc` and call:
```python
API_URL = 'https://ewds.climate.copernicus.eu/api'
API_KEY = '<your-key>'   # or use ~/.cdsapirc
AREA = [44, -10, 35, 5]  # [N, W, S, E] — Iberian Peninsula
YEARS = list(range(2000, 2021))
OUTPUT_DIR = 'glofas_data/'
paths = download_glofas_by_year(area=AREA, years=YEARS, output_dir=OUTPUT_DIR,
                                api_key=API_KEY, api_url=API_URL)
```

In [3]:
# === SYNTHETIC DEMO DATA ===
# Creates a temporary GloFAS-format NetCDF file with dis24 variable

TMPDIR = tempfile.mkdtemp()
NC_FILE = os.path.join(TMPDIR, 'glofas_demo_2000.nc')

rng = np.random.default_rng(42)
times = pd.date_range('2000-01-01', '2000-12-31', freq='D')
lats  = np.arange(44.0, 34.9, -0.05)   # 0.05° grid, N→S
lons  = np.arange(-10.0, 5.1,  0.05)   # 0.05° grid, W→E

# Synthetic discharge field: larger values for rivers (random spatial pattern)
base = rng.uniform(10, 300, (len(times), len(lats), len(lons))).astype('float32')

ds_glofas = xr.Dataset(
    {'dis24': (['time', 'latitude', 'longitude'], base)},
    coords={
        'time':      times,
        'latitude':  lats,
        'longitude': lons,
    }
)
ds_glofas['dis24'].attrs.update({
    'long_name': 'River discharge in the last 24 hours',
    'units': 'm3 s-1',
})
ds_glofas.to_netcdf(NC_FILE)
print(f'Synthetic GloFAS NetCDF created: {NC_FILE}')
print(ds_glofas)

# Point of interest
LAT = 39.5
LON = -3.5

Synthetic GloFAS NetCDF created: /var/folders/44/dw7p6q9108xcc4mmh_f7q1vc0000gn/T/tmpqea7nsmh/glofas_demo_2000.nc
<xarray.Dataset> Size: 81MB
Dimensions:    (time: 366, latitude: 183, longitude: 302)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2000-01-01 2000-01-02 ... 2000-12-31
  * latitude   (latitude) float64 1kB 44.0 43.95 43.9 43.85 ... 35.0 34.95 34.9
  * longitude  (longitude) float64 2kB -10.0 -9.95 -9.9 -9.85 ... 4.95 5.0 5.05
Data variables:
    dis24      (time, latitude, longitude) float32 81MB 234.4 137.3 ... 177.6


---
## 1. Reading downloaded NetCDF files

In [4]:
# Extract discharge at a specific coordinate from a .nc file
df_glofas = read_glofas_nc(NC_FILE, lat=LAT, lon=LON)
df_glofas = df_glofas.set_index('date')
print(df_glofas.describe())
df_glofas.head()

        discharge
count  366.000000
mean   158.553680
std     83.129402
min     10.970588
25%     88.985165
50%    162.603111
75%    232.278473
max    299.431519


,discharge
date,
2000-01-01,175.367935
2000-01-02,32.689964
2000-01-03,209.493286
2000-01-04,289.463043
2000-01-05,195.581635


---
## 2. Visualisation

In [5]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df_glofas.index, df_glofas['discharge'], lw=0.6, color='darkorange')
axes[0].set_ylabel('GloFAS discharge (m³/s)')
axes[0].set_title(f'GloFAS v4.0 · ({LAT}°N, {LON}°E) — synthetic demo', fontsize=13)
axes[0].grid(alpha=0.3)

monthly = df_glofas['discharge'].groupby(df_glofas.index.month).mean()
axes[1].bar(monthly.index, monthly.values, color='darkorange', alpha=0.85)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1].set_ylabel('Mean discharge (m³/s)')
axes[1].set_title('Seasonal regime')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/GloFAS_download.png', dpi=100, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/GloFAS_download.png')

Plot saved to /tmp/GloFAS_download.png
